In [85]:
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

price = df[['Price']].values

from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

df_boost = df.copy()
df_boost["Lag1"] = df_boost["Price"].shift(1)
df_boost["Diff"] = df_boost["Price"] - df_boost["Lag1"]
df_boost = df_boost.dropna()

split = int(len(df_boost) * 0.8)

train = df_boost.iloc[:split]
test = df_boost.iloc[split:]

X_train = train[["Lag1"]]
y_train = train["Diff"]

X_test = test[["Lag1"]]
y_test = test["Diff"]

actual = test["Price"].values

# ======================
# GB
# ======================
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_final = X_test["Lag1"].values + gb_pred

# ======================
# XGB
# ======================
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_final = X_test["Lag1"].values + xgb_pred

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n=== {name} ===")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R2: {r2:.6f}")
    print(f"MAPE: {mape:.6f}")

evaluate("GB", actual, gb_final)
evaluate("XGB", actual, xgb_final)




=== GB ===
MAE: 948.164455
RMSE: 1195.599535
R2: 0.996632
MAPE: 1.125514

=== XGB ===
MAE: 597.442307
RMSE: 897.808825
R2: 0.998101
MAPE: 0.687401


In [88]:
os.makedirs("models", exist_ok=True)

joblib.dump(gb, "models/gradient_boosting.pkl")


['models/gradient_boosting.pkl']

In [89]:
joblib.dump(xgb, "models/xgboost.pkl")

['models/xgboost.pkl']